# Coral / Reef Segmentation — U-Net on SUIM (Colab)

**Runtime:** `Runtime → Change runtime type → T4 GPU`.

**Data:** SUIM (Islam et al., IROS 2020) — https://irvlab.cs.umn.edu/resources/suim-dataset.
Easiest: copy the file ID from the official SUIM Google Drive link (the part between `/d/` and `/view`)
into `SUIM_GDRIVE_FILE_ID` in the config cell — the zip (~4.9 GB) downloads straight into Colab.
Alternatively upload the zip(s) to `MyDrive/suim/` and leave the ID empty. Any zip layout works: the
notebook finds the `images/` + `masks/` folders inside and keeps `TEST` separate.

**What this model can and cannot do**
- SUIM labels 8 classes per pixel: water, divers, plants/sea-grass, wrecks, robots,
  **reefs & invertebrates (RI)**, fish & vertebrates, sea-floor & rocks.
- So the model learns **where reef is** → coral presence + coral cover %.
- SUIM has **no healthy / bleached labels**. Locally, bleaching is still judged by colour, but only
  *inside* the predicted reef area (instead of the whole image). Learning bleaching itself would need a
  separate healthy-vs-bleached dataset.

**Quality gate:** reef (RI) IoU on the official SUIM TEST split ≥ `GATE_REEF_IOU`. If it fails, the last cell
says so and the local project keeps the colour heuristic (nothing is swapped in silently).

**Output** (download from `MyDrive/suim/` into the local project):
- `suim_unet.pt` → `models/coral_segmenter/suim_unet.pt`
- `suim_unet_config.json` + `suim_test_metrics.json` → `models/coral_segmenter/`

In [ ]:
!nvidia-smi
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
# ---- Config -------------------------------------------------------------
ENCODER   = 'resnet34'     # ImageNet-pretrained encoder
IMG_H, IMG_W = 320, 480    # multiples of 32; SUIM frames are ~4:3 / 3:2
EPOCHS    = 60
PATIENCE  = 12             # early stop on val mIoU
BATCH     = 12
LR        = 3e-4
SEED      = 42
VAL_FRAC  = 0.1            # carved from train_val; TEST is never touched until the end
GATE_REEF_IOU = 0.50

# Data source. EITHER paste the Google Drive file ID of the official SUIM zip (the long string
# between /d/ and /view in its share link) to download it straight into Colab (~1 min, no upload),
# OR leave it empty and put the SUIM zip(s) in MyDrive/suim/ yourself.
SUIM_GDRIVE_FILE_ID = ''
# -------------------------------------------------------------------------

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, glob
DRIVE_DIR = '/content/drive/MyDrive/suim'     # trained model + metrics are saved here
os.makedirs(DRIVE_DIR, exist_ok=True)

if SUIM_GDRIVE_FILE_ID.strip():
    !pip install -q gdown
    SUIM_URL = f'https://drive.google.com/uc?id={SUIM_GDRIVE_FILE_ID.strip()}'
    !gdown "{SUIM_URL}" -O /content/SUIM.zip
    zips = ['/content/SUIM.zip']
    assert os.path.exists(zips[0]) and os.path.getsize(zips[0]) > 1e8, (
        'Download failed (Drive sometimes blocks very popular files: "too many users"). Retry later, '
        'or open the link, "Add shortcut to Drive"/download it, put the zip in MyDrive/suim/ and clear the ID.')
else:
    zips = sorted(glob.glob(f'{DRIVE_DIR}/*.zip'))
assert zips, 'No SUIM zip: set SUIM_GDRIVE_FILE_ID in the config cell, or upload the zip(s) to MyDrive/suim/'
for z in zips:
    print('extracting', os.path.basename(z), f'({os.path.getsize(z) / 1e9:.1f} GB)')
    with zipfile.ZipFile(z) as f:
        f.extractall('/content/suim')

## Find image / mask pairs and decode masks
SUIM masks are RGB-coded with each channel either 0 or 255 (threshold at 128):
BW 000 · HD 001 · PF 010 · WR 011 · RO 100 · **RI 101** · FV 110 · SR 111 (bits = R G B).

In [ ]:
import numpy as np, cv2, random, json
from pathlib import Path

CLASSES = ['water', 'divers', 'plants_seagrass', 'wrecks', 'robots', 'reef_invertebrates', 'fish_vertebrates', 'seafloor_rocks']
REEF = CLASSES.index('reef_invertebrates')
# (R,G,B) bits -> class index
CODE_TO_CLASS = {(0,0,0): 0, (0,0,1): 1, (0,1,0): 2, (0,1,1): 3, (1,0,0): 4, (1,0,1): 5, (1,1,0): 6, (1,1,1): 7}

def decode_mask(path):
    m = cv2.imread(str(path), cv2.IMREAD_COLOR)[:, :, ::-1]  # -> RGB
    bits = (m >= 128).astype(np.uint8)
    idx = bits[..., 0] * 4 + bits[..., 1] * 2 + bits[..., 2]
    lut = np.zeros(8, np.uint8)
    for (r, g, b), c in CODE_TO_CLASS.items():
        lut[r * 4 + g * 2 + b] = c
    return lut[idx]

def pairs_under(root):
    out = []
    for img_dir in Path(root).rglob('images'):
        mask_dir = img_dir.parent / 'masks'
        if not mask_dir.is_dir():
            continue
        masks = {p.stem: p for p in mask_dir.iterdir() if p.suffix.lower() in ('.bmp', '.png', '.jpg')}
        for p in img_dir.iterdir():
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png') and p.stem in masks:
                out.append((p, masks[p.stem]))
    return out

all_pairs = pairs_under('/content/suim')
test_pairs = [pr for pr in all_pairs if 'TEST' in [part.upper() for part in pr[0].parts]]
trainval_pairs = [pr for pr in all_pairs if pr not in test_pairs]
assert test_pairs and trainval_pairs, 'Could not find both train_val and TEST image/mask folders'
random.Random(SEED).shuffle(trainval_pairs)
n_val = int(len(trainval_pairs) * VAL_FRAC)
val_pairs, train_pairs = trainval_pairs[:n_val], trainval_pairs[n_val:]
print(f'train {len(train_pairs)} | val {len(val_pairs)} | TEST {len(test_pairs)}')

m = decode_mask(train_pairs[0][1]); print('classes in first mask:', [CLASSES[i] for i in np.unique(m)])

In [ ]:
import torch, albumentations as A
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(SEED)

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = A.Compose([
    A.Resize(IMG_H, IMG_W), A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.ColorJitter(0.2, 0.2, 0.2, 0.03, p=0.5), A.Normalize(MEAN, STD),
])
eval_tf = A.Compose([A.Resize(IMG_H, IMG_W), A.Normalize(MEAN, STD)])

class SUIM(Dataset):
    def __init__(self, pairs, tf):
        self.pairs, self.tf = pairs, tf
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        img_p, mask_p = self.pairs[i]
        img = cv2.imread(str(img_p))[:, :, ::-1]
        mask = decode_mask(mask_p)
        if mask.shape != img.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        o = self.tf(image=img, mask=mask)
        return torch.from_numpy(o['image'].transpose(2, 0, 1)).float(), torch.from_numpy(o['mask']).long()

dl_train = DataLoader(SUIM(train_pairs, train_tf), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
dl_val = DataLoader(SUIM(val_pairs, eval_tf), batch_size=BATCH, num_workers=2)
dl_test = DataLoader(SUIM(test_pairs, eval_tf), batch_size=BATCH, num_workers=2)

## Train U-Net (ResNet-34 encoder) — Dice + cross-entropy loss, best checkpoint by val mIoU

In [ ]:
import segmentation_models_pytorch as smp
device = 'cuda'
model = smp.Unet(ENCODER, encoder_weights='imagenet', classes=len(CLASSES)).to(device)
dice = smp.losses.DiceLoss('multiclass')
ce = torch.nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

def confusion(loader):
    model.eval()
    cm = torch.zeros(len(CLASSES), len(CLASSES), dtype=torch.long, device=device)
    with torch.no_grad(), torch.autocast('cuda'):
        for x, y in loader:
            p = model(x.to(device)).argmax(1)
            y = y.to(device)
            cm += torch.bincount(y.flatten() * len(CLASSES) + p.flatten(), minlength=len(CLASSES) ** 2).view(len(CLASSES), -1)
    return cm.cpu().numpy()

def iou_from_cm(cm):
    inter = np.diag(cm); union = cm.sum(0) + cm.sum(1) - inter
    return np.where(union > 0, inter / np.maximum(union, 1), np.nan)

CKPT = f'{DRIVE_DIR}/suim_unet.pt'
best_miou, bad = -1, 0
for epoch in range(EPOCHS):
    model.train(); total = 0
    for x, y in dl_train:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.autocast('cuda'):
            out = model(x); loss = dice(out, y) + ce(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item()
    sched.step()
    ious = iou_from_cm(confusion(dl_val)); miou = np.nanmean(ious)
    print(f'epoch {epoch+1:3d}  loss {total/len(dl_train):.3f}  val mIoU {miou:.3f}  reef IoU {ious[REEF]:.3f}')
    if miou > best_miou:
        best_miou, bad = miou, 0
        torch.save(model.state_dict(), CKPT)
    else:
        bad += 1
        if bad >= PATIENCE:
            print('early stop'); break
print('best val mIoU', round(best_miou, 3))

## Evaluate on the official SUIM TEST split (never used for training or model selection)

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device))
cm = confusion(dl_test)
ious = iou_from_cm(cm)
pixel_acc = float(np.diag(cm).sum() / cm.sum())
# Binary reef-vs-rest view (what the local project actually uses)
tp = cm[REEF, REEF]; fp = cm[:, REEF].sum() - tp; fn = cm[REEF, :].sum() - tp
reef_iou = float(tp / max(1, tp + fp + fn))
metrics = {
    'dataset': 'SUIM TEST (official split)', 'model': f'U-Net {ENCODER}', 'input_hw': [IMG_H, IMG_W],
    'n_test_images': len(test_pairs), 'pixel_accuracy': round(pixel_acc, 4),
    'mean_iou': round(float(np.nanmean(ious)), 4),
    'per_class_iou': {c: (None if np.isnan(v) else round(float(v), 4)) for c, v in zip(CLASSES, ious)},
    'reef_iou': round(reef_iou, 4),
    'reef_precision': round(float(tp / max(1, tp + fp)), 4),
    'reef_recall': round(float(tp / max(1, tp + fn)), 4),
    'gate_reef_iou': GATE_REEF_IOU, 'passed_gate': reef_iou >= GATE_REEF_IOU,
}
print(json.dumps(metrics, indent=2))
json.dump(metrics, open(f'{DRIVE_DIR}/suim_test_metrics.json', 'w'), indent=2)
json.dump({'encoder': ENCODER, 'classes': CLASSES, 'reef_class': REEF, 'input_hw': [IMG_H, IMG_W],
           'mean': MEAN, 'std': STD}, open(f'{DRIVE_DIR}/suim_unet_config.json', 'w'), indent=2)

In [ ]:
# Visual check: 4 TEST images, ground truth vs prediction (reef = magenta)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 3, figsize=(12, 12))
for row, (img_p, mask_p) in enumerate(random.Random(1).sample(test_pairs, 4)):
    img = cv2.imread(str(img_p))[:, :, ::-1]
    x = torch.from_numpy(eval_tf(image=img)['image'].transpose(2, 0, 1))[None].float().to(device)
    with torch.no_grad():
        pred = model(x).argmax(1)[0].cpu().numpy()
    gt = cv2.resize(decode_mask(mask_p), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST)
    for ax, im, t in zip(axes[row], [cv2.resize(img, (IMG_W, IMG_H)), gt == REEF, pred == REEF], ['image', 'GT reef', 'pred reef']):
        ax.imshow(im); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/suim_test_examples.png', dpi=110)

In [ ]:
if metrics['passed_gate']:
    print(f"PASS: reef IoU {metrics['reef_iou']:.3f} >= {GATE_REEF_IOU}. Download suim_unet.pt, "
          "suim_unet_config.json and suim_test_metrics.json from MyDrive/suim into models/coral_segmenter/.")
else:
    print(f"FAIL: reef IoU {metrics['reef_iou']:.3f} < {GATE_REEF_IOU}. Options before switching dataset: "
          "ENCODER='resnet50' or 'efficientnet-b3', IMG_H/W=384/576, EPOCHS=100. If still below the gate, "
          "keep the colour heuristic locally and report this result.")